In [ ]:
from pathlib import Path
import os

OUT_DIR = Path("./Econometrics").resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

USE_SYNTHETIC_DATA = False

START_DATE = "2015-01-01"
END_DATE   = None  # None = today
FREQ       = "D"   # 'D' daily, 'W' weekly, 'M' month-end

FRED_TARGET_SERIES = "SP500"                 # target (prices → returns)
FRED_X_SERIES      = ["NASDAQCOM", "DJIA", "VIXCLS"]
YF_TICKERS         = ["AAPL", "MSFT"]        # more X’s (prices → returns)
FRED_API_KEY       = os.environ.get("FRED_API_KEY", None)

# Modeling
P_MAX, Q_MAX = 3, 3      # small, fast grid
H_EXPOST = 5             # ex-post horizon
H_EXANTE = 2             # ex-ante horizon

print("Outputs ->", OUT_DIR)


Outputs will be saved to: /Users/jjburrell/Econometrics/Econometrics/econ425_outputs


In [2]:
import numpy as np, pandas as pd
from pandas_datareader import data as pdr
import yfinance as yf

def to_dt(s): 
    return pd.to_datetime(s) if s is not None else pd.Timestamp.today()
start, end = to_dt(START_DATE), to_dt(END_DATE)

def fred(series_id):
    kw = {"data_source":"fred","start":start,"end":end}
    if FRED_API_KEY: kw["api_key"]=FRED_API_KEY
    return pdr.DataReader(series_id, **kw).rename(series_id)

def yf_close(ticker):
    df = yf.download(ticker, start=start, end=end, progress=False)
    if df.empty: raise RuntimeError(f"No data: {ticker}")
    return df["Close"].rename(ticker)

if not USE_SYNTHETIC_DATA:
    # Target + Xs (prices)
    target_px = fred(FRED_TARGET_SERIES)
    fred_x = []
    for sid in FRED_X_SERIES:
        try: fred_x.append(fred(sid))
        except Exception as e: print("FRED fail:", sid, e)
    fred_df = pd.concat([target_px] + fred_x, axis=1)

    yfs = []
    for t in YF_TICKERS:
        try: yfs.append(yf_close(t))
        except Exception as e: print("YF fail:", t, e)
    yf_df = pd.concat(yfs, axis=1) if len(yfs) else pd.DataFrame()

    all_px = pd.concat([fred_df, yf_df], axis=1).sort_index().resample(FREQ).last().ffill()
    log_px = np.log(all_px)
    ret = log_px.diff().dropna()
    # final dataset: y + 4 X (minimum)
    cols = [(FRED_TARGET_SERIES, "y_return")] + [(sid, f"X_{sid}") for sid in FRED_X_SERIES] + [(t, f"X_{t}") for t in YF_TICKERS]
    keep = cols[:1+4]
    df = ret[[c[0] for c in keep]].copy()
    df.columns = [c[1] for c in keep]
else:
    # Synthetic (fast, no internet)
    n = 220; idx = pd.date_range("2015-01-01", periods=n, freq="D")
    rng = np.random.default_rng(42)
    def ar1(phi, sigma, n):
        x=np.zeros(n); e=rng.normal(0,sigma,n)
        for t in range(1,n): x[t]=phi*x[t-1]+e[t]
        return x
    X1=ar1(0.5,0.01,n); X2=ar1(0.3,0.015,n); X3=ar1(0.6,0.02,n); X4=ar1(0.4,0.01,n)
    u=rng.normal(0,0.01,n); eps=np.zeros(n)
    for t in range(1,n): eps[t]=0.6*eps[t-1]+u[t]+0.7*u[t-1]
    y=0.2*X1-0.1*X2+0.15*X3+0.1*X4+eps
    df = pd.DataFrame({"y_return":y,"X_market":X1,"X_rates":X2,"X_vol":X3,"X_sector":X4}, index=idx)

# checks + save dataset
assert df.shape[1] >= 5 and df.shape[0] >= 100
df.index.name = "date"
dta_path = OUT_DIR/"project_dataset.dta"; csv_path = OUT_DIR/"project_dataset.csv"
df.to_stata(dta_path, write_index=True); df.to_csv(csv_path)
print("Saved:", dta_path); print("Saved:", csv_path)
df.head()


TypeError: 'str' object is not callable